# 05 · Hyper-parameter Tuning
Systematic tuning (3-fold cross-validation, scoring = ROC-AUC).

**Reads:** `data/processed/train.parquet`, `model/feature_config.joblib`  
**Writes:** `model/best_params.json`

In [ ]:
import os, json, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_recall_curve, roc_curve,
                             confusion_matrix, classification_report, f1_score)
from sklearn.calibration import calibration_curve

from pathlib import Path

# Repo layout (notebooks live in <repo>/notebooks/, everything else is resolved from the repo root):
#   data/raw        <- original Kaggle csv files (flights.csv, airlines.csv, airports.csv)
#   data/processed  <- intermediate files passed from one notebook to the next
#   model/          <- saved models / configs / final bundle
#   outputs/        <- small reports (csv tables)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
MODEL_DIR = ROOT / "model"
OUT_DIR = ROOT / "outputs"
for d in (DATA_RAW, DATA_PROC, MODEL_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42


In [ ]:
cfg = joblib.load(MODEL_DIR / "feature_config.joblib")
FEATURES, TARGET = cfg["features"], cfg["target"]
train = pd.read_parquet(DATA_PROC / "train.parquet")
print("train:", train.shape, "| features:", len(FEATURES))

### Helpers

In [ ]:
def best_threshold(y, p):
    """Probability threshold that maximises F1."""
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    return thr[np.argmax(f1[:-1])]

def get_model(name):
    if name == "Logistic Regression":
        return Pipeline([("scale", StandardScaler()), ("clf", LogisticRegression(max_iter=500))])
    if name == "Random Forest":
        return Pipeline([("clf", RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=50,
                                                        n_jobs=-1, random_state=SEED))])
    return Pipeline([("clf", HistGradientBoostingClassifier(random_state=SEED))])

MODELS = ["Logistic Regression", "Random Forest", "HistGradientBoosting"]


### Search

In [ ]:
cv = StratifiedKFold(3, shuffle=True, random_state=SEED)
searches = {
    "Logistic Regression": (GridSearchCV(get_model("Logistic Regression"), {"clf__C": [0.01, 0.1, 1, 10]},
                                         scoring="roc_auc", cv=cv), 150_000),
    "Random Forest": (RandomizedSearchCV(get_model("Random Forest"),
                                         {"clf__n_estimators": [100, 200], "clf__max_depth": [8, 12, 16],
                                          "clf__min_samples_leaf": [20, 50, 100]},
                                         n_iter=5, scoring="roc_auc", cv=cv, random_state=SEED), 60_000),
    "HistGradientBoosting": (RandomizedSearchCV(get_model("HistGradientBoosting"),
                                                {"clf__learning_rate": [0.03, 0.05, 0.1], "clf__max_iter": [200, 400],
                                                 "clf__max_leaf_nodes": [15, 31, 63], "clf__min_samples_leaf": [50, 100, 200],
                                                 "clf__l2_regularization": [0, 0.1, 1]},
                                                n_iter=10, scoring="roc_auc", cv=cv, random_state=SEED), 150_000),
}
best_params = {}
for name, (search, n) in searches.items():
    sub = train.sample(min(n, len(train)), random_state=SEED)
    t0 = time.time(); search.fit(sub[FEATURES], sub[TARGET])
    best_params[name] = {k.replace("clf__", ""): v for k, v in search.best_params_.items()}
    print(f"{name:22s} CV ROC-AUC = {search.best_score_:.4f} | {best_params[name]} | {time.time()-t0:.0f}s")

In [ ]:
with open(MODEL_DIR / "best_params.json", "w") as f:
    json.dump(best_params, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
print("Saved", MODEL_DIR / "best_params.json")